# 17 · Optimización y escalabilidad
Mide una optimización (unión con y sin broadcast) y la prueba de escalabilidad del pipeline (subsecciones 3.6 y 3.7).

In [ ]:
import os
import sys
import json

from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
EVIDENCIAS = BASE / "evidencias"

EVIDENCIAS.mkdir(parents=True, exist_ok=True)

HDFS_BASE = (
    "hdfs://localhost:9000"
    "/user/junior/datalake_afp_v2"
)

derby_path = (
    BASE / "metastore" / "metastore_db"
).as_posix()

os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["HADOOP_CONF_DIR"] = (
    "/opt/hadoop-3.3.6/etc/hadoop"
)
os.environ["YARN_CONF_DIR"] = os.environ["HADOOP_CONF_DIR"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

spark = (
    SparkSession.builder
    .appName("AFP_V2_Benchmark_Rendimiento")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.ui.port", "4052")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        HDFS_BASE + "/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("=== CONEXIÓN SPARK ===")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print(
    "Catálogo:",
    spark.conf.get("spark.sql.catalogImplementation")
)

print("\n=== BASES DE DATOS ===")

spark.sql("SHOW DATABASES").show()

In [ ]:
import time
import json
import statistics
import subprocess
from pathlib import Path
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
EVIDENCIAS = BASE / "evidencias"
EVIDENCIAS.mkdir(parents=True, exist_ok=True)

print("=== ENTORNO DEL BENCHMARK ===")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))
print("Paralelismo por defecto:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

print("\n=== RECURSOS DE LA MAQUINA ===")

subprocess.run(["free", "-h"], check=True)

print("\n=== TABLAS ORIGINALES ===")

tablas = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "sbs_indicadores_adicionales",
    "tabla_analitica_mensual"
]

conteos = {
    nombre: spark.table(f"afp_analytics.{nombre}").count()
    for nombre in tablas
}

print("Tablas:", len(conteos))
print("Filas acumuladas:", sum(conteos.values()))

assert sum(conteos.values()) == 110407

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "false")

cuotas = (
    spark.table("afp_analytics.sbs_valor_cuota")
    .select(
        "fecha",
        "afp",
        "fondo",
        "valor_cuota",
        "fecha_mes"
    )
)

mensual = (
    spark.table("afp_analytics.tabla_analitica_mensual")
    .select(
        "fecha_mes",
        "rent_real_integra_12m"
    )
)

print("=== TAMAÑO DE LAS FUENTES ===")
print("Valor cuota:", cuotas.count())
print("Tabla mensual:", mensual.count())

print("\n=== ESQUEMAS ===")
cuotas.printSchema()
mensual.printSchema()

In [ ]:
def consulta_sin_broadcast():

    return (
        cuotas.join(
            mensual,
            on="fecha_mes",
            how="inner"
        )
        .withColumn(
            "valor_ponderado",
            F.col("valor_cuota")
            * F.col("rent_real_integra_12m")
        )
        .groupBy("afp", "fondo")
        .agg(
            F.count("*").alias("registros"),
            F.sum("valor_ponderado").alias("total_ponderado")
        )
    )


def consulta_con_broadcast():

    return (
        cuotas.join(
            broadcast(mensual),
            on="fecha_mes",
            how="inner"
        )
        .withColumn(
            "valor_ponderado",
            F.col("valor_cuota")
            * F.col("rent_real_integra_12m")
        )
        .groupBy("afp", "fondo")
        .agg(
            F.count("*").alias("registros"),
            F.sum("valor_ponderado").alias("total_ponderado")
        )
    )

In [ ]:
print("=== ANTES: UNION SIN BROADCAST ===")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

consulta_sin_broadcast().explain(True)

print("\n=== DESPUES: UNION CON BROADCAST ===")

consulta_con_broadcast().explain(True)

In [ ]:
def ejecutar_y_resumir(construir):

    df = construir()

    filas = df.orderBy("afp", "fondo").collect()

    return [
        (
            r["afp"],
            r["fondo"],
            r["registros"],
            r["total_ponderado"]
        )
        for r in filas
    ]


def medir(nombre, construir, repeticiones=3):

    print("\nPROBANDO:", nombre)

    # Calentamiento
    referencia = ejecutar_y_resumir(construir)

    tiempos = []

    for i in range(repeticiones):

        inicio = time.perf_counter()

        resultado = ejecutar_y_resumir(construir)

        segundos = time.perf_counter() - inicio

        assert len(resultado) == len(referencia)

        tiempos.append(segundos)

        print(
            f"Repetición {i + 1}: "
            f"{segundos:.3f} segundos"
        )

    return {
        "estrategia": nombre,
        "tiempos_segundos": tiempos,
        "mediana_segundos": statistics.median(tiempos),
        "resultado": referencia
    }


spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

antes = medir(
    "Shuffle join sin broadcast",
    consulta_sin_broadcast
)

despues = medir(
    "Broadcast join",
    consulta_con_broadcast
)

print("\n=== COMPARACIÓN FINAL ===")

print(
    "Tiempo antes:",
    round(antes["mediana_segundos"], 3),
    "segundos"
)

print(
    "Tiempo después:",
    round(despues["mediana_segundos"], 3),
    "segundos"
)

mejora_pct = (
    (antes["mediana_segundos"] - despues["mediana_segundos"])
    / antes["mediana_segundos"] * 100
)

print("Variación porcentual:", round(mejora_pct, 2), "%")

In [ ]:
import math

resultado_antes = sorted(
    antes["resultado"],
    key=lambda x: (str(x[0]), str(x[1]))
)

resultado_despues = sorted(
    despues["resultado"],
    key=lambda x: (str(x[0]), str(x[1]))
)

assert len(resultado_antes) == len(resultado_despues)

for a, b in zip(resultado_antes, resultado_despues):

    assert a[:3] == b[:3]

    assert math.isclose(
        a[3] or 0.0,
        b[3] or 0.0,
        rel_tol=1e-9,
        abs_tol=1e-6
    )

print("RESULTADOS EQUIVALENTES: SI")
print("Grupos comparados:", len(resultado_antes))

In [ ]:
reporte_optimizacion = {
    "fecha": datetime.now().isoformat(),
    "spark": spark.version,
    "master": spark.sparkContext.master,
    "tecnica": "broadcast join",
    "tiempos_antes_segundos": antes["tiempos_segundos"],
    "tiempos_despues_segundos": despues["tiempos_segundos"],
    "mediana_antes_segundos": antes["mediana_segundos"],
    "mediana_despues_segundos": despues["mediana_segundos"],
    "variacion_pct": mejora_pct,
    "resultados_equivalentes": True
}

ruta = EVIDENCIAS / "17_optimizacion_broadcast.json"

with ruta.open("w", encoding="utf-8") as f:
    json.dump(reporte_optimizacion, f, indent=2)

print("Evidencia:", ruta)